# Google Earth Engine Lesson: Selecting an Exact Country Boundary

## Introduction

When working with satellite data in Google Earth Engine (GEE), one of the first steps is defining the **study area**.

For example, if we want to analyze Sentinel-2 images over Germany, we need to tell GEE:

> "Use only the area inside Germany's official boundary."

A common beginner mistake is to create a rectangle around the country using coordinates. Although this works technically, it does not represent the real country shape.

---

# 1. The Problem with Using a Rectangle

A simple way to define Germany is:

```javascript
var countryArea = ee.Geometry.Polygon([
  [
    [5.87,47.27],
    [15.04,47.27],
    [15.04,55.06],
    [5.87,55.06],
    [5.87,47.27]
  ]
]);
```

This creates a rectangular area:

```
+----------------------+
|                      |
|      Germany         |
|                      |
+----------------------+
```

However, Germany is not a rectangle.

The rectangle also includes parts of neighboring countries:

* France
* Switzerland
* Austria
* Czech Republic
* Poland
* Denmark

Therefore, if we use:

```javascript
.clip(countryArea)
```

GEE will clip the image to the rectangle, not the actual German border.

---

# 2. The Better Approach: Use Administrative Boundaries

For country-scale remote sensing projects, we normally use a global boundary dataset.

These datasets already contain official country borders.

Examples:

* FAO GAUL
* USDOS LSIB
* Natural Earth

Instead of manually drawing Germany, we simply select Germany from the dataset.

---

# 3. Load Country Boundaries

One simple dataset available in GEE is:

```javascript
USDOS/LSIB_SIMPLE/2017
```

We load it as a FeatureCollection:

```javascript
var countries = ee.FeatureCollection(
  "USDOS/LSIB_SIMPLE/2017"
);
```

A FeatureCollection is a collection of geographic features.

In this case:

```
FeatureCollection

├── Germany
├── France
├── Italy
├── Spain
├── Iran
├── Japan
└── ...
```

---

# 4. Select Germany

Now we filter the dataset and keep only Germany:

```javascript
var germany = countries.filter(
  ee.Filter.eq('country_na','Germany')
);
```

Explanation:

`filter()` means:

> Keep only features that match a condition.

The condition is:

```javascript
country_na = Germany
```

The result is:

```
World boundaries

        ↓

Germany boundary only
```

---

# 5. Display the Boundary

To check our study area:

```javascript
Map.centerObject(germany,6);

Map.addLayer(
  germany,
  {},
  "Germany boundary"
);
```

Now GEE displays the real country shape:

```
        ______
      /        \
     | Germany |
      \________/

```

instead of a rectangle.

---

# 6. Use the Boundary in Sentinel-2 Analysis

After creating the Germany boundary, we can use it in satellite image processing.

Example:

## Load Sentinel-2 data

```javascript
var collection = ee.ImageCollection(
  "COPERNICUS/S2_SR_HARMONIZED"
)
.filterBounds(germany)
.filterDate(
  '2024-06-01',
  '2024-08-31'
);
```

Here:

```javascript
.filterBounds(germany)
```

means:

> Keep only Sentinel-2 images that cover Germany.

---

## Clip the final image

```javascript
var medianComposite = collection
  .median()
  .clip(germany);
```

The output image now covers only Germany.

---

# 7. Complete Example

```javascript
// =====================================
// 1. Load country boundaries
// =====================================

var countries = ee.FeatureCollection(
  "USDOS/LSIB_SIMPLE/2017"
);


// =====================================
// 2. Select Germany
// =====================================

var germany = countries.filter(
  ee.Filter.eq('country_na','Germany')
);


// =====================================
// 3. Load Sentinel-2
// =====================================

var collection = ee.ImageCollection(
  "COPERNICUS/S2_SR_HARMONIZED"
)
.filterBounds(germany)
.filterDate(
  '2024-06-01',
  '2024-08-31'
);


// =====================================
// 4. Create composite
// =====================================

var medianComposite = collection
  .median()
  .clip(germany);


// =====================================
// 5. Visualization
// =====================================

var trueColor = {
  bands:['B4','B3','B2'],
  min:0,
  max:3000
};


// =====================================
// 6. Display
// =====================================

Map.centerObject(germany,6);

Map.addLayer(
  medianComposite,
  trueColor,
  "Germany Sentinel-2 Composite"
);

Map.addLayer(
  germany,
  {},
  "Germany Boundary"
);
```

---

# Key Concepts to Remember

## 1. Geometry

A manually created shape.

Example:

```javascript
ee.Geometry.Polygon()
```

Useful for:

* lakes
* farms
* small study areas
* watersheds

---

## 2. FeatureCollection

A collection of geographic features.

Example:

```javascript
ee.FeatureCollection()
```

Useful for:

* countries
* provinces
* administrative regions

---

## 3. filter()

Used to select specific features.

Example:

```javascript
.filter(
ee.Filter.eq('country_na','Germany')
)
```

Meaning:

> Select the feature where the country name equals Germany.

---

## 4. clip()

Cuts an image to the study area.

Example:

```javascript
image.clip(germany)
```

Meaning:

> Keep only pixels inside Germany.

---

# Practical Rule

For GEE projects:

| Study Area                      | Recommended Method                  |
| ------------------------------- | ----------------------------------- |
| Country (Germany, Iran, Turkey) | Use administrative boundary dataset |
| Province/State                  | Use administrative boundary dataset |
| Lake                            | Draw polygon manually               |
| Field/Farm                      | Upload shapefile or draw polygon    |
| Watershed                       | Use watershed dataset               |

---

# Final Takeaway

For accurate remote sensing analysis:

**Do not use rectangles for countries.**

Instead:

```
Country boundary dataset
          ↓
Select country
          ↓
Use as study area
          ↓
Analyze satellite data
```

This produces scientifically correct results and is the standard workflow used in GIS and remote sensing research.
